<a href="https://colab.research.google.com/github/koderlad/M507D---Methods-of-Prediction/blob/main/M507D_Week_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Importing Dependencies**

In [2]:
import pandas as pd
import sklearn
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
from sklearn.neural_network import MLPClassifier

## **Loading Dataset**

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/m-mahdavi/teaching/refs/heads/main/datasets/mnist.csv')

### **Splitting Dataset**

In [4]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(['class', 'id'], axis=1), df['class'], test_size=0.2, stratify=df['class'])

## **Quick Exploration**

In [5]:
print(f"Train Data Shape: {X_train.shape} \nTest Data Shape: {X_test.shape}")

Train Data Shape: (3200, 784) 
Test Data Shape: (800, 784)


In [6]:
X_train.dropna(inplace=True)
X_train.drop_duplicates(inplace=True)
X_test.dropna(inplace=True)
X_test.drop_duplicates(inplace=True)

In [7]:
print(f"Train Data Shape: {X_train.shape} \nTest Data Shape: {X_test.shape}")

Train Data Shape: (3200, 784) 
Test Data Shape: (800, 784)


Training and Test dataset does not have null or duplicate values.

### **Class Balance**

In [8]:
y_train.value_counts()

,count
class,
1,389
7,341
3,333
8,333
6,313
2,312
0,301
4,295
9,293


Kind of balanced, "Accuracy" can be used as evaluation metric.

## **Baseline Model**

I am using the same classifier but without Hyper-Parameter finetuning, just the default values it already comes assigned with. And no Scaling of the values.

In [9]:
baseline = MLPClassifier()

### **Cross Validating**
Making sure that the model's accuracy isn't a one time thing and isn't a good memorizer but a good classfier.

In [10]:
cv_score_baseline = cross_val_score(baseline, X_train, y_train, cv=5, scoring='accuracy')

In [11]:
print(f"Cross Val Score (Baseline): {cv_score_baseline}")
print(f"Average Accuracy: {cv_score_baseline.mean()}\n Standard Deviation: {cv_score_baseline.std()}")

Cross Val Score (Baseline): [0.865625  0.8703125 0.8578125 0.8859375 0.8609375]
Average Accuracy: 0.868125
 Standard Deviation: 0.009862333648787207


### **Actual Training**

In [12]:
baseline.fit(X_train, y_train)

MLPClassifier()

In [13]:
y_pred_baseline = baseline.predict(X_test)

In [14]:
baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
print(f"Baseline Model Accuracy on Test Data is {round(baseline_accuracy, 4)}")

Baseline Model Accuracy on Test Data is 0.87


## **Now with Hyper-parameter fine-tuning**

In [24]:
params = {"MLP": {
    # 'model__hidden_layer_sizes': [(100,), (50, 50)],
    'model__activation': ['relu', 'tanh'],
    'model__solver': ['lbfgs', 'adam'],
    'model__learning_rate_init': [0.001, 0.01],
    # 'model__max_iter': [200, 400, 700],
    'model__tol': [0.0001, 0.001, 0.01],
    'model__early_stopping': [True],
    'model__validation_fraction': [0.1, 0.15]
}}

In [16]:
pipe = Pipeline(steps=[
    ('standardscaler', StandardScaler()),
    ('model', baseline)
])

### **With Scaling Features**

### **Cross Validating**

In [17]:
cv_score_scaled = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')

In [18]:
print(f"Cross Val Score (Baseline): {cv_score_scaled}")
print(f"Average Accuracy: {cv_score_scaled.mean()}\n Standard Deviation: {cv_score_scaled.std()}")

Cross Val Score (Baseline): [0.921875  0.921875  0.91875   0.9296875 0.9140625]
Average Accuracy: 0.92125
 Standard Deviation: 0.005096720759468783


### **Actual Training**

In [19]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('model', MLPClassifier())])

In [20]:
y_pred_scaled = pipe.predict(X_test)

In [21]:
scaled_accuracy = accuracy_score(y_test, y_pred_scaled)
print(f"Accuracy (Scaled Features) on Test Data is {round(scaled_accuracy, 4)}")

Accuracy (Scaled Features) on Test Data is 0.9113


## **Searching the best Hyperparameters**

In [25]:
search = GridSearchCV(
    estimator = pipe,
    param_grid = params['MLP'],
    scoring = 'accuracy',
    cv = 5,
    n_jobs = -1,
)

In [26]:
search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('standardscaler', StandardScaler()),
                                       ('model', MLPClassifier())]),
             n_jobs=-1,
             param_grid={'model__activation': ['relu', 'tanh'],
                         'model__early_stopping': [True],
                         'model__learning_rate_init': [0.001, 0.01],
                         'model__solver': ['lbfgs', 'adam'],
                         'model__tol': [0.0001, 0.001, 0.01],
                         'model__validation_fraction': [0.1, 0.15]},
             scoring='accuracy')

In [27]:
print("Best Score (Accuracy): ", round(search.best_score_, 4))
print("Best Params: ", search.best_params_)

Best Score (Accuracy):  0.9169
Best Params:  {'model__activation': 'relu', 'model__early_stopping': True, 'model__learning_rate_init': 0.001, 'model__solver': 'lbfgs', 'model__tol': 0.0001, 'model__validation_fraction': 0.1}


In [28]:
beast_model = search.best_estimator_

In [29]:
y_pred = beast_model.predict(X_test)

In [30]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy on Test Data is {round(accuracy,4)}")

Accuracy on Test Data is 0.8975


## **Comparison**

In [31]:
cr_baseline = classification_report(y_test, y_pred_baseline)
cr_scaled = classification_report(y_test, y_pred_scaled)
cr = classification_report(y_test, y_pred)

In [32]:
print(cr_baseline, cr_scaled, cr)

              precision    recall  f1-score   support

           0       0.92      0.92      0.92        75
           1       0.98      1.00      0.99        97
           2       0.84      0.90      0.87        78
           3       0.86      0.75      0.80        84
           4       0.82      0.86      0.84        74
           5       0.77      0.75      0.76        73
           6       0.92      0.97      0.94        78
           7       0.88      0.87      0.88        85
           8       0.87      0.86      0.86        83
           9       0.79      0.78      0.79        73

    accuracy                           0.87       800
   macro avg       0.87      0.87      0.87       800
weighted avg       0.87      0.87      0.87       800
               precision    recall  f1-score   support

           0       0.90      0.92      0.91        75
           1       0.95      0.99      0.97        97
           2       0.92      0.90      0.91        78
           3       0.90 